# 03 — Оценка сохранённого checkpoint

Запускайте сверху вниз в чистом kernel. Обучение не запускается; pretrained-веса повторно не скачиваются. Порог всегда 0.5 и не подбирается на validation.

## Как интерпретировать результат
Это классификация окон вокруг **известных отметок**, а не автоматический поиск события во всём видео.
Согласованный `within_room_unique_video` split: 30 train / 10 validation; точные копии не пересекаются.
Общий кабинет, люди и неизвестные сеансы могут пересекаться. Независимость камер и сеансов не подтверждена.
Validation используется для выбора checkpoint по loss, а не как независимый финальный тест.
Первый smoke-run предсказывал нарушение для всех validation-окон (accuracy 0.5, recall 1.0).
Это проверка работоспособности, не хорошее качество модели и не законченный детектор.
Исторические метрики и provenance сохранены в `experiments/r3d18_smoke_result.md` и прежних `runs/`.

In [ ]:
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "src/overfitted_underqualified").is_dir() and (p / "notebooks").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Запустите из корня репозитория или notebooks/")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import os

import pandas as pd
from IPython.display import display

from overfitted_underqualified.ml.evaluate import evaluate_prepared, prepare_evaluation
from overfitted_underqualified.ml.plots import plot_confusion, show_windows


## 1. Checkpoint и сохранённая конфигурация

По умолчанию открывается первый smoke-run. Для нового эксперимента задайте CHECKPOINT_PATH либо измените путь ниже. На Linux/Brev скопируйте checkpoint и видео. Пути к текущим данным задаются явно, поэтому старые абсолютные Windows-пути не мешают восстановлению. Содержимое сверяется с сохранёнными хешами.

In [ ]:
checkpoint_path = Path(os.environ.get("CHECKPOINT_PATH", str(ROOT / "runs/r3d18_smoke/best.pt")))
if not checkpoint_path.is_absolute():
    checkpoint_path = ROOT / checkpoint_path
evaluation = prepare_evaluation(
    checkpoint_path, device=os.environ.get("EVAL_DEVICE", "cpu"),
    events_path=ROOT / "data/annotations/events.csv",
    split_path=ROOT / "data/processed/split_proposal.csv",
    video_dir=Path(os.environ.get("VIDEO_DIR", str(ROOT / "data/raw/videos"))),
)
display(evaluation["checkpoint"]["config"])
print("Checkpoint:", checkpoint_path, "epoch:", evaluation["checkpoint"]["epoch"])
print("Проверено validation-окон:", len(evaluation["dataset"]))


## 2. Оценка validation

Модель работает в eval mode: параметры и BatchNorm не обновляются. Precision/recall/F1 относятся к violation=1. Accuracy считается по всем окнам. Оценка не перезаписывает результаты обучения.

In [ ]:
import time

import torch

from overfitted_underqualified.ml.evaluate import aggregate_video_predictions

if evaluation["device"].type == "cuda":
    torch.cuda.synchronize(evaluation["device"])
processing_start = time.perf_counter()
metrics, rows = evaluate_prepared(evaluation)
if evaluation["device"].type == "cuda":
    torch.cuda.synchronize(evaluation["device"])
processing_sec = time.perf_counter() - processing_start
predictions = pd.DataFrame(rows)
display(pd.Series({key: value for key, value in metrics.items() if key != "confusion_matrix"}, name="validation"))
plot_confusion(metrics)
display(predictions)

# Distinguish window metrics above from canonical-video metrics below.
video_metrics, video_rows = aggregate_video_predictions(rows)
video_predictions = pd.DataFrame(video_rows)
display(pd.Series({key: value for key, value in video_metrics.items() if key != "confusion_matrix"}, name="video-level validation"))
plot_confusion(video_metrics)
display(video_predictions)
print(f"Оценка окон: {processing_sec:.3f} с; среднее {processing_sec / len(video_rows):.3f} с/видео.")
print("Включены decode, preprocessing, transfer, inference и оконные метрики; загрузка checkpoint исключена.")
print("Это время выбранных окон, не полного просмотра видео; warm-up не выполнялся.")


## 3. Просмотр ошибок

Показываем до шести ошибочно классифицированных окон с фиксированным порогом. Это разбор ошибок, а не изменение разметки или настройка порога.

In [ ]:
errors = predictions.loc[predictions["label"].ne(predictions["prediction"])]
display(errors)
selected = errors.head(6)
show_windows(evaluation["dataset"], selected.index,
             [f"{row.clip_id}; y={row.label}, pred={row.prediction}, p={row.probability:.2f}"
              for row in selected.itertuples()])
print("Всего ошибок:", len(errors), "из", len(predictions))


Следующий эксперимент должен проверять конкретную гипотезу. Для вывода о поиске событий нужен отдельный full-video протокол; для переноса на другие камеры/сеансы — независимые данные.

## 4. False positives и false negatives
Разделяем лишние тревоги и пропуски. Для каждого окна доступны исходные ID и время центра.
Показываем обе группы отдельно: пустая группа означает отсутствие таких ошибок на этой выборке,
а не доказательство безошибочности. Кадры — визуальные свидетельства для исследования, не юридическое подтверждение.

In [ ]:
error_analysis = predictions.copy()
error_analysis["outcome"] = [
    "TP" if truth == pred == 1 else "TN" if truth == pred == 0 else "FP" if pred == 1 else "FN"
    for truth, pred in zip(error_analysis["label"], error_analysis["prediction"])
]
error_analysis["source_ids"] = [list(record.aliases) for record, _ in evaluation["dataset"].samples]
display(error_analysis.groupby("outcome").agg(windows=("clip_id", "size"), mean_probability=("probability", "mean")))
for outcome in ("FP", "FN"):
    subset = error_analysis.loc[error_analysis["outcome"].eq(outcome)]
    print(outcome, "окон:", len(subset))
    display(subset)
    selected = subset.head(3)
    show_windows(evaluation["dataset"], selected.index,
                 [f"{outcome}: {row.clip_id}; p={row.probability:.2f}" for row in selected.itertuples()])


## 5. Временная диагностика: что измеряется сейчас
Модель выдаёт вероятность класса окна, **не predicted timestamp**. Поэтому timestamp MAE,
ошибка локализации и доля событий в допуске здесь **N/A**, а не ноль.
`center_sec` задан разметкой для положительных окон; сравнивать его с разметкой как предсказание нельзя.
Ниже проверяем только выборку: границы окна, ближайший реально выбранный кадр и его отклонение
от отметки. Это погрешность дискретизации кадров, не ошибка модели. Для FP истинной отметки нет.
Для настоящей оценки локализации нужны full-video inference и заранее заданное сопоставление событий,
включая обработку лишних и пропущенных событий. Контракт FastAPI `Finding.timecode_sec` сохраняется;
он не заполняется центрами известных окон в качестве якобы найденных событий.

In [ ]:
import numpy as np

from overfitted_underqualified.ml.dataset import sample_times

timing_rows = []
dataset = evaluation["dataset"]
for index, (record, center) in enumerate(dataset.samples):
    requested = sample_times(center, record.last_time, dataset.window_sec, dataset.frames)
    sampled = np.clip(np.rint(requested * record.fps), 0, record.frame_count - 1) / record.fps
    reference = center if record.label == 1 else np.nan
    nearest = sampled[np.argmin(abs(sampled - reference))] if record.label == 1 else np.nan
    timing_rows.append({"clip_id": record.clip_id, "outcome": error_analysis.loc[index, "outcome"],
        "reference_sec": reference, "window_start_sec": sampled.min(), "window_end_sec": sampled.max(),
        "nearest_sample_sec": nearest, "sampling_offset_sec": abs(nearest-reference),
        "predicted_timestamp_sec": np.nan, "timestamp_error_sec": np.nan})
timing = pd.DataFrame(timing_rows)
display(timing)
display(timing["sampling_offset_sec"].describe())
print("Timestamp localization metrics: N/A — no model timestamp predictions")


## 6. Выводы и следующий исследовательский шаг
Сначала прочитайте итог ниже для выбранного checkpoint. Затем запишите наблюдения по FP/FN:
видно ли действие после crop, хватает ли временного контекста, похожи ли обычные действия на нарушения.
Это гипотезы для следующего эксперимента, а не автоматическая коррекция источника меток.

In [ ]:
counts = error_analysis["outcome"].value_counts()
print(f"Validation: {len(predictions)} окон; FP={counts.get('FP', 0)}, FN={counts.get('FN', 0)}.")
print(f"Accuracy={metrics['accuracy']:.3f}, precision={metrics['precision']:.3f}, recall={metrics['recall']:.3f}, F1={metrics['f1']:.3f}.")
if predictions["prediction"].nunique() == 1:
    print("Модель выдаёт один класс для всех окон. Проверьте дисбаланс предсказаний; recall отдельно недостаточен.")
print("Локализация не оценена. Независимость камер/сеансов не подтверждена; это не готовый detector.")


Перед коммитом очистите outputs, execution counts и attachments. Не переносите изображения,
описания событий или индивидуальные предсказания в публичные отчёты. Сохранённые runs остаются локальными.

## Сохранение воспроизводимого отчёта
CLI `python -m overfitted_underqualified.ml.evaluate --checkpoint <best.pt> --output-dir <новый reports/каталог>`
сохраняет video/window predictions, confusion matrix, JSON с checkpoint SHA-256, конфигурацией,
манифестом split и измерением времени. Это отдельный оценочный проход без обучения.
Для первого checkpoint отчёт и разбор FP/FN находятся в `reports/r3d18_smoke_evaluation/`.
Video-level score — максимум вероятностей окон канонического видео при пороге 0.5.
На текущей validation одно окно на видео, поэтому оконные и video-level метрики совпадают.
